# Impedance ML Engineering — Session 01
## Understand and audit the impedance matrices
**Core: 60 minutes · Foundations · Optional extension: 20–40 minutes**

## Why this matters
What does a matrix contain, and which patterns are measurement artifacts versus plausible research questions?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Audit the patient → implant → visit → directed-cell hierarchy.
- Mask invalid measurements without treating missingness as zero.
- Reconstruct a correctly ordered 12×12 matrix and interpret its spatial profile.
- Distinguish descriptive patterns from predictive and causal claims.

## R-to-Python / prerequisite bridge
`read.csv` → `pd.read_csv`; `left_join` → `.merge(validate="many_to_one")`; `pivot_wider` → `.pivot`; `NA` → `np.nan`; R indices start at 1, NumPy positions at 0.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

## Concept and API reminders
A matrix cell is a measurement, a visit is a repeated observation, and a patient is an independent sampling unit under this fixture. A large cell table can conceal a small independent sample. Exploratory plots generate questions; they do not by themselves test a hypothesis or demonstrate predictive value.

Useful imports: `from pathlib import Path`, `import pandas as pd`, `import numpy as np`, `import matplotlib.pyplot as plt`. `pd.read_csv(path)` returns a DataFrame. `frame.merge(other,on=key,validate="many_to_one",indicator=True)` can expose key mismatches. `frame.pivot(index=...,columns=...,values=...)` raises on duplicate cells, which is preferable to hiding them by averaging.

## Task 1 — Load the data and predict the grains (5 min)
Import pathlib, pandas, NumPy, and Matplotlib. Read `patients.csv`, `visits.csv`, and `matrix_cells.csv` from `data/ci_telemetry/`. Before running, predict which table has the most rows and why. Report shape, dtypes, and unique patient/visit counts. Acceptance: 96 patients, 360 visits, 51, 840 cells. Hint: `Path("data/ci_telemetry") / "visits.csv"`.

## Task 2 — Check keys before joining (6 min)
Assert unique patient IDs, visit IDs, and `(visit_id, stim_contact, record_contact)` keys; each visit has 144 records. Join visits to patients with `validate="many_to_one"` and inspect unmatched keys with `indicator=True`. Then join visit current onto cells, also many-to-one. Acceptance: the cell join still has 51, 840 rows and no unmatched visit. Explain why the rows are not independent patients.

## Task 3 — Define usable cells and units (7 min)
Keep the original columns. Define `usable` as QC `ok` AND finite voltage. Create `z_kohm = voltage_mv / stim_current_ua` only where usable, otherwise NaN. API: `np.isfinite`, `.eq`, `.where`. Check current is positive. Report valid, missing, and saturated counts; show why `dropna()` alone retains some invalid readings. Derive why mV/µA equals kΩ.

## Task 4 — Rebuild one directed matrix (7 min)
Use visit `CI001_M00`. Pivot stimulation contacts to rows and recording contacts to columns, then explicitly reindex both axes to `range(1,13)`. Predict the shape before `.to_numpy()`. Acceptance: 12×12, missing cells remain NaN, the array [0, 1] matches stim 1/record 2. Explain why taking only one triangle or calling this a covariance matrix loses information. Hint: use `pivot`, not an aggregation that hides duplicate keys.

## Task 5 — Plot a labeled heatmap (7 min)
Display that matrix with `imshow`, `origin="upper"`, a colorbar in kΩ, and ticks labeled 1–12. Label x as recording contact and y as stimulating contact. Show an explicit missing-data color using a copied colormap and `set_bad`. Interpret diagonal versus off-diagonal structure and name one pattern that cannot identify a clinical condition.

## Task 6 — Summarize spatial distance with coverage (7 min)
For the selected visit define index distance `abs(stim_contact-record_contact)`. Calculate mean proxy and valid count per distance, and plot the profile. Report diagonal (distance 0), near (1), and far (≥6) means. Hint: `.groupby(...).agg(...)`; average both directions as distinct measured entries. Do not interpret index distance as millimetres.

## Task 7 — Explore repeated measurements (6 min)
Load `visit_features.csv` as supplied observed summaries. Plot trajectories for six explicitly selected patients against `months_since_implant`; color or annotate protocol. Count patients at windows 0, 1, 3, 6: expect 96, 96, 96, 72. Explain why 360 visits do not mean 360 independent participants and why the missing last window matters.

## Task 8 — Write an analysis contract (7 min)
Write a six-sentence note covering the measurement definition, observation hierarchy, QC policy, one descriptive result, one statistical question, and one ML question. State the shared future target: next scheduled visit diagonal mean from current data for an unseen patient. Keep scientific outcomes such as hearing performance separate: they are absent from this fixture.

## Retrieval practice
Without copying earlier code, select QC-valid diagonal cells from one different visit and calculate mean plus count. Predict the maximum count first.

## Professional reasoning
1. Why can a finite saturated value be more dangerous than a missing value?
2. What additional keys would bilateral implants or technical repeats require?
3. What could go wrong if channel order is assumed rather than documented?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Compare two visits with shared heatmap limits and plot their difference only on cells valid at both visits. Document how changing protocol complicates interpretation.

## API references
[scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)